# Data quality check

## Import neccesary lib

In [1]:
import pandas as pd
from pathlib import Path


## Merging into 1 Dataframe to easily check

In [2]:
path = Path("../data/raw/online_retail_II.xlsx")
dict_of_df = pd.read_excel(path, sheet_name=None)
df = pd.concat(dict_of_df.values(), ignore_index=True)


## Check missing values

In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [6]:
print((df.isnull().sum() / len(df)) * 100) 

Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64


### So in this dataframe, we have 4382 rows of description and 243k rows of customer_id are null. Which are 0.41% of Description and nearly 22.8% of customer ID are null. And i'm using RFM modeling so i only keep the customer ID are valid

## Checking duplicates

In [8]:
print(df.duplicated().sum())

34335


### About 34k rows is duplicated. Need to be deduplicate in staging

## Validity rows

### Overall descriptive statistics

In [13]:
print(df[['Quantity', 'Price']].describe().round(3))

          Quantity        Price
count  1067371.000  1067371.000
mean         9.939        4.649
std        172.706      123.553
min     -80995.000   -53594.360
25%          1.000        1.250
50%          3.000        2.100
75%         10.000        4.150
max      80995.000    38970.000


### Check Price anomalies

In [23]:
print(f"Rows with Price = 0 (Free items): {len(df[df['Price'] == 0])}")
print(f"Rows with Price < 0 (Errors): {len(df[df['Price'] < 0])}")

Rows with Price = 0 (Free items): 6202
Rows with Price < 0 (Errors): 5


#### So with the Price = 0, i keep them like a promotional for customer, but with the price < 0, i drop because they are not valid sales

### Check Quantity anomalies

In [22]:
print(f"Rows with Quantity < 0: {len(df[df['Quantity'] < 0])}")

Rows with Quantity < 0: 22950


### Check InvoiceNo: Find Cancelled Orders (contain 'C')

In [26]:
cancelled_orders = df[df['Invoice'].astype(str).str.contains('C', na=False)]
print(f"Total cancelled orders (Invoice starts with 'C'): {len(cancelled_orders)}\n")

Total cancelled orders (Invoice starts with 'C'): 19494



### Check StockCode: Find non-product codes (letters only, no numbers)

In [27]:
strange_stockcodes = df[df['StockCode'].astype(str).str.contains('^[a-zA-Z]+$', regex=True, na=False)]
print("--- Special/Non-Product StockCodes ---")
print(strange_stockcodes['StockCode'].value_counts().head(10))

--- Special/Non-Product StockCodes ---
StockCode
POST         2122
DOT          1446
M            1421
D             177
S             104
ADJUST         67
AMAZONFEE      43
DCGSSGIRL      25
DCGSSBOY       23
PADS           19
Name: count, dtype: int64


### Ensure InvoiceDate is in datetime format

In [33]:
df['InvoiceDate'].info()
print("\n")
print(f"First Invoice Date: {df['InvoiceDate'].min()}")
print(f"Last Invoice Date: {df['InvoiceDate'].max()}")

<class 'pandas.Series'>
RangeIndex: 1067371 entries, 0 to 1067370
Series name: InvoiceDate
Non-Null Count    Dtype         
--------------    -----         
1067371 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 8.1 MB


First Invoice Date: 2009-12-01 07:45:00
Last Invoice Date: 2011-12-09 12:50:00
